# A³ - Algorithms, Adapting Features, Adjusting Hyperparameters

## QUA³CK-Phase

Die iterative A³-Schleife umfasst:

1. **Algorithm selection** - geeignete Algorithmen auswählen,
2. **Adapting features** - Merkmale fachlich begründet anpassen,
3. **Adjusting hyperparameters** - Modellparameter abstimmen.

## Umsetzung im Projekt

Das PV-Projekt vergleicht eine einfache Median-Baseline mit einem
`HistGradientBoostingRegressor`. Das nichtlineare Modell kann Wechselwirkungen
zwischen Einstrahlung, Temperatur, Bewölkung, Feuchte und Wind abbilden.
Monotoniebedingungen stabilisieren die interaktive Szenarioanalyse.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 30)


In [ ]:
import matplotlib.pyplot as plt

from pv_weather import (
    MODEL_FEATURES,
    TARGET,
    add_features,
    load_project_data,
    train_yield_model,
)
from pv_weather.features import MONOTONIC_CONSTRAINTS
from pv_weather.modeling import (
    MAX_MODEL_SOLAR_ZENITH_DEG,
    MIN_MODEL_GLOBAL_RADIATION_J_CM2,
)

data, source = load_project_data(
    ROOT / "data" / "processed" / "hourly_pv_weather.csv"
)
featured = add_features(data)
print(source)


## A1 - Algorithm selection


In [ ]:
algorithm_choice = pd.DataFrame(
    [
        (
            "Median-Baseline",
            "Referenz ohne Wetterwissen",
            "Prüft, ob ML überhaupt Mehrwert liefert",
        ),
        (
            "Histogram Gradient Boosting",
            "Nichtlineare Regression mit Interaktionen und Monotoniebedingungen",
            "Produktives Projektmodell",
        ),
    ],
    columns=["Algorithmus", "Eigenschaft", "Rolle im Projekt"],
)
display(algorithm_choice)


Der Testdatensatz besteht aus den neuesten 20 % der verwendbaren
PV-relevanten Stunden. Eine zufällige Aufteilung würde Vergangenheit und
Zukunft vermischen und die zeitliche Übertragbarkeit zu optimistisch bewerten.


## A2 - Adapting features


In [ ]:
feature_contract = pd.DataFrame(
    [
        ("temperature_c", "°C", "Lufttemperatur", "direkt"),
        ("relative_humidity_pct", "%", "Feuchte", "direkt"),
        ("global_radiation_j_cm2", "J/cm²", "Energieangebot der Sonne", "direkt"),
        ("cloud_cover_oktas", "Achtel", "Bewölkung", "direkt"),
        ("wind_speed_m_s", "m/s", "Kühlung", "direkt"),
        (
            "estimated_module_temperature_c",
            "°C",
            "NOCT-artige Modultemperaturnäherung",
            "aus Temperatur, Strahlung und Wind",
        ),
        (
            "diffuse_share",
            "Anteil",
            "Zusammensetzung der Strahlung",
            "Diffusstrahlung / Globalstrahlung",
        ),
    ],
    columns=["Merkmal", "Einheit", "Fachliche Rolle", "Entstehung"],
)
display(feature_contract)
assert feature_contract["Merkmal"].tolist() == MODEL_FEATURES

preview_columns = [
    "temperature_c",
    "global_radiation_j_cm2",
    "wind_speed_m_s",
    "estimated_module_temperature_c",
    "diffuse_share",
    "thermal_stress_c",
    TARGET,
]
display(featured[preview_columns].head())


Uhrzeit, Monat und Sonnenstand werden zwar für Exploration und Filterung
berechnet, sind aber bewusst keine Eingaben des meteorologischen
Prognosemodells. Die installierte Leistung dient ausschließlich zur Bildung
der Zielvariable und zur späteren absoluten Skalierung.


## A3 - Adjusting hyperparameters


In [ ]:
final_hyperparameters = pd.Series(
    {
        "learning_rate": 0.07,
        "max_iter": 190,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 30,
        "l2_regularization": 1.0,
        "random_state": 42,
        "test_fraction": 0.2,
        "minimale Globalstrahlung": MIN_MODEL_GLOBAL_RADIATION_J_CM2,
        "maximaler Sonnenzenit": MAX_MODEL_SOLAR_ZENITH_DEG,
    },
    name="Projektwert",
)
display(final_hyperparameters.to_frame())

constraint_names = {-1: "nicht steigend", 0: "frei", 1: "nicht fallend"}
constraints = pd.Series(
    {
        feature: constraint_names[value]
        for feature, value in MONOTONIC_CONSTRAINTS.items()
    },
    name="Isolierte Modellrichtung",
)
display(constraints.to_frame())


Die Parameter sind die aktuell im Projekt festgelegte Konfiguration. Eine
vollständige Hyperparametersuche mit separatem Validierungsfenster ist ein
sinnvoller Ausbaupunkt; der Testzeitraum darf dabei nicht zur Optimierung
verwendet werden.


In [ ]:
bundle = train_yield_model(data)

experiment_result = pd.DataFrame(
    {
        "MAE": [
            bundle.metrics["baseline_mae"],
            bundle.metrics["model_mae"],
        ],
        "RMSE": [
            bundle.metrics["baseline_rmse"],
            bundle.metrics["model_rmse"],
        ],
        "R²": [
            bundle.metrics["baseline_r2"],
            bundle.metrics["model_r2"],
        ],
    },
    index=["Median-Baseline", "HistGradientBoosting"],
)
display(experiment_result.round(4))
print(f"Zeitlicher Test ab: {bundle.split_timestamp}")


In [ ]:
importance = (
    pd.Series(bundle.feature_importance, name="Permutation Importance")
    .sort_values()
)
importance.plot.barh(figsize=(9, 4.5), color="#E19A18")
plt.title("Merkmalsbeitrag im zeitlichen Test")
plt.xlabel("Zunahme des Fehlers nach Permutation")
plt.tight_layout()
plt.show()


## Technische Verankerung der A³-Phase

| A³-Schritt | Umsetzung |
|---|---|
| Algorithm selection | `DummyRegressor` und `HistGradientBoostingRegressor` in `pv_weather/modeling.py` |
| Adapting features | `add_features` und `estimate_module_temperature` in `pv_weather/features.py` |
| Adjusting hyperparameters | explizite Modellkonfiguration in `train_yield_model` |
| Reproduzierbarkeit | feste Zufallszahl, zentrale Merkmalsliste und gemeinsame Kernlogik |
| Experimentauswertung | Metriken und Permutationswichtigkeit im `YieldModelBundle` |

## Übergabe an C

Die C-Phase bewertet Baseline und Modell quantitativ sowie qualitativ. Sie
prüft außerdem, ob die Ergebnisse die Forschungsfrage belastbar beantworten
und welche Einschränkungen kommuniziert werden müssen.
